## Setup Environment

### Dimensions API:

In [1]:
# Connect With Drive
# %cd to api key

import os

os.getcwd()

'/home/aklinux/Documents/MIT-Research/D-Index Research/King Code/Dimensions_Diversity_Index-20240601T161145Z-001/Dimensions_Diversity_Index'

In [23]:
import dimcli
from dimcli.shortcuts import dslquery_json as dslquery
from dotenv import load_dotenv
import os
import matplotlib.pyplot as plt
import pandas as pd

# Load environment variables from .env
load_dotenv()

api_key = os.getenv("DIMENSIONS_API_KEY")
endpoint="https://app.dimensions.ai"

if api_key is None:
    raise ValueError("API key not found. Make sure to set DIMENSIONS_API_KEY in your .env file.")

dimcli.login(key=api_key, endpoint=endpoint)
dsl = dimcli.Dsl()

Dimcli - Dimensions API Client (v1.6)
Connected to: <https://app.dimensions.ai/api/dsl> - DSL v2.14
Method: manual login


In [25]:
seed_query = """
search publications
where year = 2025
  and count(research_org_countries) > 1
  and count(authors) >= 3
  and count(authors) <= 20
  and times_cited >= 2
return publications[
    id
    +title
    +year
    +doi
    +journal
    +times_cited
    +research_org_countries
    +authors
    +concepts
    +concepts_scores
] sort by times_cited desc
limit 500
"""

seed_candidates = dsl.query(seed_query)
seed_df = seed_candidates.as_dataframe()

seed_df.head()

Returned Publications: 500 (total = 262092)
Time: 14.83s


,id,title,authors,concepts,concepts_scores,doi,research_org_countries,times_cited,year,journal.id,journal.title
0,pub.1137313608,Sustainable data analysis with Snakemake,"[{'affiliations': [{'city': 'Essen', 'city_id'...","[command line tool, scripting language, intera...","[{'concept': 'command line tool', 'relevance':...",10.12688/f1000research.29032.2,"[{'id': 'SG', 'name': 'Singapore'}, {'id': 'DK...",1609,2025,jour.1491941,F1000Research
1,pub.1188983313,Transparency in the Reporting of Artificial In...,"[{'affiliations': [{'city': None, 'city_id': N...","[artificial intelligence, human oversight, AI ...","[{'concept': 'artificial intelligence', 'relev...",10.70389/pjs.100082,"[{'id': 'NZ', 'name': 'New Zealand'}, {'id': '...",1533,2025,jour.1486861,Premier Journal of Science
2,pub.1188983310,Revised Surgical CAse REport (SCARE) Guideline...,"[{'affiliations': [{'city': 'Cambridge', 'city...","[surgical case reports, nine-point Likert scal...","[{'concept': 'surgical case reports', 'relevan...",10.70389/pjs.100079,"[{'id': 'NZ', 'name': 'New Zealand'}, {'id': '...",815,2025,jour.1486861,Premier Journal of Science
3,pub.1185871559,Global patterns and trends in breast cancer in...,"[{'affiliations': [{'city': 'Lyon', 'city_id':...","[female breast cancer, cancer incidence, World...","[{'concept': 'female breast cancer', 'relevanc...",10.1038/s41591-025-03502-3,"[{'id': 'US', 'name': 'United States'}, {'id':...",505,2025,jour.1113716,Nature Medicine
4,pub.1183108622,World Stroke Organization: Global Stroke Fact ...,"[{'affiliations': [{'city': 'Auckland', 'city_...","[World Stroke Organization, non-communicable d...","[{'concept': 'World Stroke Organization', 'rel...",10.1177/17474930241308142,"[{'id': 'AT', 'name': 'Austria'}, {'id': 'CA',...",485,2025,jour.1036847,International Journal of Stroke


In [26]:
import networkx as nx
from itertools import combinations

G = nx.Graph()

for _, row in seed_df.iterrows():
    authors = row["authors"]
    
    author_ids = []
    
    if isinstance(authors, list):
        for a in authors:
            if not isinstance(a, dict):
                continue
            
            # Prefer stable researcher ID if available
            rid = a.get("researcher_id") or a.get("id")
            
            if rid is not None:
                author_ids.append(str(rid))
            else:
                # Fallback to name if researcher_id missing
                first = a.get("first_name")
                last = a.get("last_name")
                if first and last:
                    author_ids.append(f"{first.strip()} {last.strip()}")

    # Remove duplicates and None safely
    author_ids = list(set([x for x in author_ids if x is not None]))

    # Only add edges if at least 2 authors
    if len(author_ids) >= 2:
        for a, b in combinations(author_ids, 2):
            G.add_edge(a, b)

print("Total authors (nodes):", G.number_of_nodes())
print("Total edges:", G.number_of_edges())
print("Connected components:", nx.number_connected_components(G))

if G.number_of_nodes() > 0:
    largest_component = max(nx.connected_components(G), key=len)
    print("Largest component size:", len(largest_component))

Total authors (nodes): 4062
Total edges: 20136
Connected components: 427
Largest component size: 61


In [27]:
largest_component = max(nx.connected_components(G), key=len)

print("Largest component author count:", len(largest_component))

Largest component author count: 61


In [28]:
import networkx as nx
from itertools import combinations
from collections import defaultdict

def author_key(a: dict):
    if not isinstance(a, dict):
        return None
    rid = a.get("researcher_id") or a.get("id")
    if rid:
        return f"RID:{rid}"
    first, last = a.get("first_name"), a.get("last_name")
    if first and last:
        return f"NAME:{first.strip()} {last.strip()}"
    return None

G = nx.Graph()
pub_to_authors = {}

for _, row in seed_df.iterrows():
    pub_id = row["id"]
    authors = row["authors"]

    keys = []
    if isinstance(authors, list):
        for a in authors:
            k = author_key(a)
            if k:
                keys.append(k)

    keys = sorted(set(keys))
    pub_to_authors[pub_id] = set(keys)

    if len(keys) >= 2:
        for u, v in combinations(keys, 2):
            G.add_edge(u, v)

components = list(nx.connected_components(G))
print("Components:", len(components))
print("Largest size:", max(len(c) for c in components))


Components: 427
Largest size: 61


In [29]:
comp_pub_counts = []
for idx, comp in enumerate(components):
    pubs = [pid for pid, aset in pub_to_authors.items() if len(aset & comp) > 0]
    comp_pub_counts.append((idx, len(comp), len(pubs)))

# Sort by #pubs then #authors
comp_pub_counts_sorted = sorted(comp_pub_counts, key=lambda x: (x[2], x[1]), reverse=True)

print("Top 10 components by pub count:")
for idx, n_auth, n_pubs in comp_pub_counts_sorted[:10]:
    print(f"comp={idx} authors={n_auth} pubs={n_pubs}")

Top 10 components by pub count:
comp=12 authors=46 pubs=7
comp=87 authors=61 pubs=5
comp=65 authors=47 pubs=4
comp=179 authors=35 pubs=4
comp=21 authors=21 pubs=4
comp=265 authors=20 pubs=4
comp=31 authors=40 pubs=3
comp=188 authors=36 pubs=3
comp=7 authors=35 pubs=3
comp=120 authors=33 pubs=3


In [30]:
best_comp_idx = 12
best_comp = components[best_comp_idx]

demo_pub_ids = [pid for pid, aset in pub_to_authors.items() if len(aset & best_comp) > 0]
demo_df = seed_df[seed_df["id"].isin(demo_pub_ids)].copy()

print("Demo pubs:", len(demo_pub_ids))
print("Demo authors:", len(best_comp))

Demo pubs: 7
Demo authors: 46


In [33]:
def get_top_concept(row):
    c = row.get("concepts", None)
    cs = row.get("concepts_scores", None)

    # Case 1: concepts is list of dicts that already contain a score
    # e.g., [{"name": "...", "score": 0.82}, ...]
    if isinstance(c, list) and len(c) > 0 and isinstance(c[0], dict):
        # find a numeric score field if present
        score_keys = ["score", "relevance", "value", "weight"]
        def score_of(d):
            for k in score_keys:
                if k in d and isinstance(d[k], (int, float)):
                    return d[k]
            return None

        scored = [(d.get("name") or d.get("concept") or d.get("id"), score_of(d)) for d in c]
        scored = [(name, s) for name, s in scored if name is not None and s is not None]
        if scored:
            return max(scored, key=lambda x: x[1])[0]

        # If dicts exist but no score field, return first name/id as fallback
        return c[0].get("name") or c[0].get("id")

    # Case 2: concepts is list of strings, concepts_scores is list of dicts with "score"
    if isinstance(c, list) and isinstance(cs, list) and len(c) > 0 and len(cs) == len(c):
        if len(cs) > 0 and isinstance(cs[0], dict):
            score_keys = ["score", "relevance", "value", "weight"]
            scores = []
            for d in cs:
                s = None
                for k in score_keys:
                    if k in d and isinstance(d[k], (int, float)):
                        s = d[k]
                        break
                scores.append(s)

            if any(s is not None for s in scores):
                top_i = max(range(len(scores)), key=lambda i: (scores[i] is not None, scores[i] if scores[i] is not None else -1))
                return c[top_i]

        # Case 3: concepts_scores is list of numbers (your original assumption)
        if len(cs) > 0 and isinstance(cs[0], (int, float)):
            top_i = max(range(len(cs)), key=lambda i: cs[i])
            return c[top_i]

    # If no concept info
    return None

demo_df["top_concept"] = demo_df.apply(get_top_concept, axis=1)
print(demo_df["top_concept"].value_counts(dropna=False).head(20))

top_concept
S-scheme heterojunction                  4
S-scheme photocatalysts                  1
sustainable soil management practices    1
photoelectrochemical water splitting     1
Name: count, dtype: int64


In [34]:
component_summaries = []

for idx, comp in enumerate(components):
    pubs = [pid for pid, aset in pub_to_authors.items() if len(aset & comp) > 0]
    comp_df = seed_df[seed_df["id"].isin(pubs)].copy()

    comp_df["top_concept"] = comp_df.apply(get_top_concept, axis=1)

    top_counts = comp_df["top_concept"].value_counts()
    dominant = top_counts.index[0] if len(top_counts) > 0 else None

    component_summaries.append({
        "comp": idx,
        "authors": len(comp),
        "pubs": len(pubs),
        "dominant_concept": dominant
    })

summary_df = pd.DataFrame(component_summaries)
summary_df.sort_values(["pubs", "authors"], ascending=False).head(15)

,comp,authors,pubs,dominant_concept
12,12,46,7,S-scheme heterojunction
87,87,61,5,oxygen evolution reaction
65,65,47,4,all-solid-state lithium-sulfur batteries
179,179,35,4,advanced oxidation processes
21,21,21,4,doped g-C3N4
265,265,20,4,S-scheme heterojunction
31,31,40,3,incidence of serious adverse events
188,188,36,3,Clinical Dementia Rating Sum
7,7,35,3,resolution of steatohepatitis
120,120,33,3,solar-driven steam generation


In [35]:
# -------- SELECT COMPONENTS --------
comp_A = 12   # Materials cluster
comp_B = 31   # Medical cluster

def extract_component(comp_idx):
    comp = components[comp_idx]
    
    pub_ids = [pid for pid, aset in pub_to_authors.items() if len(aset & comp) > 0]
    comp_df = seed_df[seed_df["id"].isin(pub_ids)].copy()
    
    return comp_df, pub_ids, comp

# Extract both
demo_A_df, demo_A_pub_ids, demo_A_authors = extract_component(comp_A)
demo_B_df, demo_B_pub_ids, demo_B_authors = extract_component(comp_B)

print("Demo A (Materials)")
print("Pubs:", len(demo_A_pub_ids))
print("Authors:", len(demo_A_authors))

print("\nDemo B (Medical)")
print("Pubs:", len(demo_B_pub_ids))
print("Authors:", len(demo_B_authors))

Demo A (Materials)
Pubs: 7
Authors: 46

Demo B (Medical)
Pubs: 3
Authors: 40


In [36]:
demo_A_df.to_csv("demo_component_12_materials_raw.csv", index=False)
demo_B_df.to_csv("demo_component_31_medical_raw.csv", index=False)

print("Saved:")
print("demo_component_12_materials_raw.csv")
print("demo_component_31_medical_raw.csv")

Saved:
demo_component_12_materials_raw.csv
demo_component_31_medical_raw.csv


In [37]:
# If you still have the original dimcli Response object for the query:
# seed_candidates = dsl.query(seed_query)

df_long = seed_candidates.as_dataframe_authors_affiliations()
df_long.head()

/home/aklinux/anaconda3/lib/python3.11/site-packages/dimcli/core/dataframe_factory.py:195: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  affiliations.fillna('', inplace=True) # 2019-09-30: simplifies subsequent operations


,aff_city,aff_city_id,aff_country,aff_country_code,aff_id,aff_name,aff_raw_affiliation,aff_state,aff_state_code,pub_id,researcher_id,first_name,last_name
0,Essen,2928810.0,Germany,DE,grid.410718.b,Essen University Hospital,"Bioinformatics and Computational Oncology, Ins...",,,pub.1137313608,ur.011436234771.81,Felix,Mölder
1,Essen,2928810.0,Germany,DE,grid.5718.b,University of Duisburg-Essen,"Bioinformatics and Computational Oncology, Ins...",,,pub.1137313608,ur.011436234771.81,Felix,Mölder
2,Essen,2928810.0,Germany,DE,grid.410718.b,Essen University Hospital,"Institute of Pathology, University Hospital Es...",,,pub.1137313608,ur.011436234771.81,Felix,Mölder
3,Essen,2928810.0,Germany,DE,grid.5718.b,University of Duisburg-Essen,"Institute of Pathology, University Hospital Es...",,,pub.1137313608,ur.011436234771.81,Felix,Mölder
4,Lausanne,2659994.0,Switzerland,CH,grid.419765.8,Swiss Institute of Bioinformatics,"Swiss Institute of Bioinformatics (SIB), Basel...",,,pub.1137313608,ur.016003101437.24,Kim Philipp,Jablonski


In [38]:
comp_A = 12
comp_B = 31

best_comp_A = components[comp_A]
best_comp_B = components[comp_B]

pub_ids_A = [pid for pid, aset in pub_to_authors.items() if len(aset & best_comp_A) > 0]
pub_ids_B = [pid for pid, aset in pub_to_authors.items() if len(aset & best_comp_B) > 0]

print("Component 12 pubs:", len(pub_ids_A))
print("Component 31 pubs:", len(pub_ids_B))

Component 12 pubs: 7
Component 31 pubs: 3


In [39]:
# If your long df has a different name, replace seed_long_df below
A_long = df_long[df_long["pub_id"].isin(pub_ids_A)].copy()
B_long = df_long[df_long["pub_id"].isin(pub_ids_B)].copy()

A_long.to_csv("demo_component_12_materials_long_raw.csv", index=False)
B_long.to_csv("demo_component_31_medical_long_raw.csv", index=False)

print("Saved demo_component_12_materials_long_raw.csv  rows:", len(A_long),
      "unique pubs:", A_long["pub_id"].nunique(),
      "unique authors:", A_long["researcher_id"].nunique(dropna=True))

print("Saved demo_component_31_medical_long_raw.csv   rows:", len(B_long),
      "unique pubs:", B_long["pub_id"].nunique(),
      "unique authors:", B_long["researcher_id"].nunique(dropna=True))

Saved demo_component_12_materials_long_raw.csv  rows: 80 unique pubs: 7 unique authors: 44
Saved demo_component_31_medical_long_raw.csv   rows: 55 unique pubs: 3 unique authors: 40


In [40]:
def make_author_template(long_df, out_name):
    # Keep one row per researcher_id if possible; fall back to name if researcher_id missing
    tmp = long_df.copy()
    tmp["author_key"] = tmp["researcher_id"].fillna(tmp["first_name"].fillna("") + " " + tmp["last_name"].fillna(""))
    authors = (tmp[["author_key", "researcher_id", "first_name", "last_name"]]
               .drop_duplicates()
               .sort_values(["last_name", "first_name"], na_position="last"))

    authors["gender"] = ""
    authors["specialization"] = ""
    authors.to_csv(out_name, index=False)
    print("Wrote", out_name, "authors:", len(authors))

make_author_template(A_long, "demo_A_author_template.csv")
make_author_template(B_long, "demo_B_author_template.csv")

Wrote demo_A_author_template.csv authors: 46
Wrote demo_B_author_template.csv authors: 40


In [41]:
pub_ids = list(set(pub_ids_A + pub_ids_B))
pub_ids_str = "[" + ",".join([f'"{pid}"' for pid in pub_ids]) + "]"

In [42]:
q = f"""
search publications
where id in {pub_ids_str}
return publications[
    id
    +title
    +year
    +doi
    +journal
    +authors
]
limit {len(pub_ids)}
"""

resp = dsl.query(q)

Returned Publications: 10 (total = 10)
Time: 0.36s


In [45]:
pub_lookup = (
    seed_df[["id", "doi", "title", "year", "journal.title"]]
    .rename(columns={"id": "pub_id", "journal.title": "journal_title"})
    .drop_duplicates()
)

In [47]:
seed_long_with_pub = df_long.merge(pub_lookup, on="pub_id", how="left")

# sanity check
print("DOI missing rate:", seed_long_with_pub["doi"].isna().mean())
seed_long_with_pub[["pub_id", "doi"]].drop_duplicates().head()

DOI missing rate: 0.0


,pub_id,doi
0,pub.1137313608,10.12688/f1000research.29032.2
33,pub.1188983313,10.70389/pjs.100082
42,pub.1188983310,10.70389/pjs.100079
51,pub.1185871559,10.1038/s41591-025-03502-3
62,pub.1183108622,10.1177/17474930241308142


In [48]:
A_long = seed_long_with_pub[seed_long_with_pub["pub_id"].isin(pub_ids_A)].copy()
B_long = seed_long_with_pub[seed_long_with_pub["pub_id"].isin(pub_ids_B)].copy()

A_long.to_csv("demo_component_12_materials_long_raw.csv", index=False)
B_long.to_csv("demo_component_31_medical_long_raw.csv", index=False)

print("A pubs:", A_long["pub_id"].nunique(), "A DOIs present:", A_long["doi"].notna().mean())
print("B pubs:", B_long["pub_id"].nunique(), "B DOIs present:", B_long["doi"].notna().mean())

A pubs: 7 A DOIs present: 1.0
B pubs: 3 B DOIs present: 1.0


In [53]:
def make_author_template_with_doi_strict(long_df, out_name):
    tmp = long_df.copy()

    # Normalize strings
    for col in ["first_name", "last_name", "aff_id", "pub_id"]:
        if col in tmp.columns:
            tmp[col] = tmp[col].astype(str).fillna("").str.strip()

    # Build strict key
    def build_key(r):
        rid = r.get("researcher_id")
        if pd.notna(rid) and str(rid).strip() not in ["", "nan", "None"]:
            return f"RID:{rid}"
        fn = r.get("first_name","").strip()
        ln = r.get("last_name","").strip()
        aff = r.get("aff_id","").strip()
        pid = r.get("pub_id","").strip()

        if fn and ln and aff and aff not in ["nan","None"]:
            return f"NAMEAFF:{fn} {ln}::{aff}"
        if fn and ln and pid:
            return f"NAMEPUB:{fn} {ln}::{pid}"
        return None

    tmp["author_key"] = tmp.apply(build_key, axis=1)
    tmp = tmp.dropna(subset=["author_key"])

    # Sort so DOI chosen deterministically
    tmp = tmp.sort_values(["author_key", "year", "doi"], na_position="last")

    author_template = (
        tmp.groupby("author_key", as_index=False)
        .first()[["author_key", "researcher_id", "first_name", "last_name", "aff_id", "aff_name", "doi"]]
    )

    author_template["gender"] = ""
    author_template["specialization"] = ""

    author_template.to_csv(out_name, index=False)
    print("Saved", out_name, "with", len(author_template), "unique authors")

make_author_template_with_doi_strict(A_long, "demo_A_author_template.csv")
make_author_template_with_doi_strict(B_long, "demo_B_author_template.csv")

Saved demo_A_author_template.csv with 46 unique authors
Saved demo_B_author_template.csv with 40 unique authors


In [11]:
# sample org: grid.258806.1
GRIDID = "grid.258806.1" #@param {type:"string"}
START_YEAR = 2019 #@param {type:"slider", min:1900, max:2020, step: 1}
END_YEAR = 2024 #@param {type:"slider", min:1900, max:2020, step: 1}

if START_YEAR < END_YEAR: START_YEAR = END_YEAR
YEARS = f"[{START_YEAR}:{END_YEAR}]"

In [17]:
pubs = dsl.query_iterative(f"""search publications
                                where year in {YEARS}
                            return publications[id+title+doi+year+type+authors+journal+issue+volume]
                            """, limit=500)

Starting iteration with limit=500 skip=0 ...
0-500 / 2730971 (0.68s)
500-1000 / 2730971 (4.56s)
1000-1500 / 2730971 (0.71s)
1500-2000 / 2730971 (4.66s)
2000-2500 / 2730971 (6.01s)
2500-3000 / 2730971 (0.90s)
3000-3500 / 2730971 (1.05s)
3500-4000 / 2730971 (1.31s)
4000-4500 / 2730971 (7.29s)
4500-5000 / 2730971 (2.44s)
5000-5500 / 2730971 (1.38s)
5500-6000 / 2730971 (1.71s)
6000-6500 / 2730971 (5.73s)
6500-7000 / 2730971 (5.07s)
7000-7500 / 2730971 (5.95s)
7500-8000 / 2730971 (2.98s)
8000-8500 / 2730971 (2.14s)
8500-9000 / 2730971 (3.14s)
9000-9500 / 2730971 (2.30s)
9500-10000 / 2730971 (5.96s)
10000-10500 / 2730971 (6.13s)
10500-11000 / 2730971 (7.03s)
11000-11500 / 2730971 (2.55s)
11500-12000 / 2730971 (6.08s)
12000-12500 / 2730971 (6.15s)
12500-13000 / 2730971 (6.64s)
13000-13500 / 2730971 (3.26s)
13500-14000 / 2730971 (6.05s)
14000-14500 / 2730971 (3.43s)
14500-15000 / 2730971 (6.31s)
15000-15500 / 2730971 (6.16s)
15500-16000 / 2730971 (5.91s)
16000-16500 / 2730971 (4.36s)
16500-170

In [22]:
print("We got", pubs.count_batch, "results out of", pubs.count_total)

We got 50000 results out of 2730971


In [2]:
# extract affiliations from a publications list
affiliations = pubs.as_dataframe_authors_affiliations()

NameError: name 'pubs' is not defined

In [23]:
affiliations.head()

,aff_city,aff_city_id,aff_country,aff_country_code,aff_id,aff_name,aff_raw_affiliation,aff_state,aff_state_code,pub_id,researcher_id,first_name,last_name
0,Warsaw,756135.0,Poland,PL,grid.440603.5,Cardinal Stefan Wyszyński University in Warsaw,"Institute of Biological Sciences, Cardinal Ste...",,,pub.1172183536,,Piotr,Kiełtyk
1,George Town,1735106.0,Malaysia,MY,grid.11875.3a,Universiti Sains Malaysia,Department of Medical Microbiology and Parasit...,,,pub.1172183526,,Kirnpal Kaur Banga,Singh
2,Kuala Selangor,1732891.0,Malaysia,MY,,Integrative Pharmacogenomics Institute (iPROMI...,Integrative Pharmacogenomics Institute (iPROMI...,Selangor,,pub.1172183526,,Mohd Zaki,Salleh
3,George Town,1735106.0,Malaysia,MY,grid.11875.3a,Universiti Sains Malaysia,Department of Medical Microbiology and Parasit...,,,pub.1172183526,,Naveed,Ahmed
4,George Town,1735106.0,Malaysia,MY,grid.11875.3a,Universiti Sains Malaysia,Department of Medical Microbiology and Parasit...,,,pub.1172183526,,Chan,Yean Yean


In [1]:
simple_df = affiliations[["pub_id", "first_name", "last_name", "aff_country", "aff_country_code", "aff_id", "aff_city", "aff_city_id", "aff_name", "aff_raw_affiliation"]]

NameError: name 'affiliations' is not defined

In [26]:
simple_df.head()

,pub_id,first_name,last_name,aff_country,aff_country_code,aff_id,aff_city,aff_city_id
0,pub.1172183536,Piotr,Kiełtyk,Poland,PL,grid.440603.5,Warsaw,756135.0
1,pub.1172183526,Kirnpal Kaur Banga,Singh,Malaysia,MY,grid.11875.3a,George Town,1735106.0
2,pub.1172183526,Mohd Zaki,Salleh,Malaysia,MY,,Kuala Selangor,1732891.0
3,pub.1172183526,Naveed,Ahmed,Malaysia,MY,grid.11875.3a,George Town,1735106.0
4,pub.1172183526,Chan,Yean Yean,Malaysia,MY,grid.11875.3a,George Town,1735106.0


In [28]:
len(simple_df)

158101

In [30]:
simple_df.dropna()

,pub_id,first_name,last_name,aff_country,aff_country_code,aff_id,aff_city,aff_city_id
0,pub.1172183536,Piotr,Kiełtyk,Poland,PL,grid.440603.5,Warsaw,756135.0
1,pub.1172183526,Kirnpal Kaur Banga,Singh,Malaysia,MY,grid.11875.3a,George Town,1735106.0
2,pub.1172183526,Mohd Zaki,Salleh,Malaysia,MY,,Kuala Selangor,1732891.0
3,pub.1172183526,Naveed,Ahmed,Malaysia,MY,grid.11875.3a,George Town,1735106.0
4,pub.1172183526,Chan,Yean Yean,Malaysia,MY,grid.11875.3a,George Town,1735106.0
...,...,...,...,...,...,...,...,...
158096,pub.1172098630,Xueguang,Zhang,China,CN,grid.413389.4,Xuzhou,1787824.0
158097,pub.1172098630,Xueguang,Zhang,China,CN,grid.13291.38,Chengdu,1815286.0
158098,pub.1172098630,Xueguang,Zhang,China,CN,grid.412901.f,Chengdu,1815286.0
158099,pub.1172098630,Yuting,Huang,China,CN,grid.413389.4,Xuzhou,1787824.0


In [35]:
simple_df.to_csv("dimensionsDataForPastYear.csv")

In [36]:
ls

 Data_extraction.ipynb           EDA.ipynb                Preprocessing.ipynb
 dimensionsDataForPastYear.csv  'List of economies.csv'


### Read PMID

In [6]:
df = pd.read_csv('Global Health/Global_health_papers_2018_2023_Pubmed.csv')
# Remove NaNs in PMID
df = df[df['PMID'].notna()]

In [7]:
df

,PMID,JournalTitle,Title,doi,Abstract,Language,Year_A,Month_A,Day_A,Year_PM,Month_PM,Day_PM,Status,MeSH,MeSH_UI,Keyword
0,34672239,Global health promotion,CORRIGENDUM: [School accreditation scheme redu...,10.1177/1757975918793277,NaN,eng,2018.0,10.0,8.0,2021,10,22,ppublish,NaN,NaN,NaN
1,33357144,Global health action,The AMR emergency: multi-sector collaboration ...,10.1080/16549716.2019.1855831,NaN,eng,NaN,NaN,NaN,2020,12,29,ppublish,Global Health|Health Policy|Humans,D014943|D006291|D006801,NaN
2,33282227,Journal of global health,How strong are Malawi's family planning progra...,10.7189/jogh.09.020901,To assess the strength of implementation of fa...,eng,NaN,NaN,NaN,2020,12,8,ppublish,Adolescent|Family Planning Services|Female|Hum...,D000293|D005193|D005260|D006801|D008295|D01539...,NaN
3,33282226,Journal of global health,Reported evidence on the effectiveness of mass...,10.7189/jogh.09.020420,An estimated 200 million women and girls in lo...,eng,NaN,NaN,NaN,2020,12,8,ppublish,Developing Countries|Family Planning Services|...,D003906|D005193|D005260|D006266|D007722|D00680...,NaN
4,33274068,Journal of global health,Health system strengthening is needed to respo...,10.7189/jogh.09.020317,NaN,eng,NaN,NaN,NaN,2020,12,5,ppublish,Delivery of Health Care|Developing Countries|H...,D003695|D003906|D000073656|D006801|D010146,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24952,33300766,International journal of health policy and man...,Implementing the UN Sustainable Development Go...,10.34172/ijhpm.2020.221,The United Nations (UN) Sustainable Developmen...,eng,2022.0,6.0,1.0,2020,12,11,epublish,Global Health|Goals|Humans|Norway|Sustainable ...,D014943|D006040|D006801|D009664|D000076502|D01...,Discourse Analysis|Framing Analysis|Nordic Reg...
24953,32945638,International journal of health policy and man...,Public Risk-Taking and Rewards During the COVI...,10.34172/ijhpm.2020.166,"Public investment, through both research grant...",eng,2022.0,5.0,1.0,2020,9,19,epublish,Adenosine Monophosphate|Alanine|Global Health|...,D000249|D000409|D014943|D006801|D058873|D01220...,Access|Biomedical R&D|Innovation|Public Value
24954,32823379,International journal of health policy and man...,Is Physical Rehabilitation Need Associated Wit...,10.34172/ijhpm.2020.150,To determine whether population-adjusted rates...,eng,2022.0,4.0,1.0,2020,8,23,epublish,Developed Countries|Global Health|Health Expen...,D019049|D014943|D005102|D006801|D007182|D00007...,Health Services Needs|Health Workforce|High-In...
24955,32668894,International journal of health policy and man...,"Beyond the Science: Advancing the ""Art and Cra...",10.34172/ijhpm.2020.131,Interesting debates are ongoing on how to deve...,eng,2022.0,3.0,1.0,2020,7,17,epublish,Capacity Building|Global Health|Humans|Impleme...,D057191|D014943|D006801|D000077626,Apprenticeship|Capacity Building|Global Health...


In [ ]:
df.shape
year = 2023 # Change this to the year you want to filter by 2018 - 2021
df = df[df['Year_PM'] == year]
df.shape

(3006, 16)

### Set the Query

To set you query you need 3 things:

1. The type of search. Could be:
* Publications
* Datasets
* Grants
* Patents
* Clinical trials
* Policy documents
* Reports
* Source titles
* Researchers
* Organizations

2. The Topic: Use any topic for your search. You can also use boolean operations such as 'and' or 'or'

3. Where: To limitate your search you can use the where to limitate by year, citations, author, and many others.

4. Columns of the resulting query: Which are the columns that we want to return from the query. Example:

| Fieldset                     | Fields                                  |
|------------------------------|-----------------------------------------|
| extras                       | altmetric date doi funders open_access pmcid pmid relative_citation_ratio research_org_cities research_org_countries research_org_country_names research_org_state_codes research_org_state_names research_orgs researchers times_cited |
| categories                   | category_bra category_for category_hra category_hrcs_hc category_hrcs_rac category_icrp_cso category_icrp_ct category_rcdc category_sdg category_uoa |
| book                         | book_doi book_series_title book_title   |
| basics                       | authors id issue journal pages title type volume year |


For more information check the [Publication's documentation](https://docs.dimensions.ai/dsl/datasource-publications.html)



In [5]:
from dimcli.shortcuts import dslquery_json as dslquery

# Function to fetch paper details by PMID
def get_paper_details(pmid):
    query = f'search publications where pmid="{pmid}" return publications[id+title+authors+year+source_title+journal+date+doi+funder_countries+funders+linkout]'
    response = dslquery(query)
    return response['publications']

# Function to fetch paper details by PMID
def get_paper_details(pmid):
    # Corrected query with proper '+' signs for field concatenation
    query = f'''
    search publications where pmid="{pmid}"
    return publications[id+title+authors+year+source_title+journal+date+doi+linkout+funder_countries+funders]
    '''
    response = dslquery(query)
    paper_details = response['publications']
    if paper_details:
        paper = paper_details[0]
        # Initialize the information dictionary
        info = {
            'ID': paper.get('id', ''),
            'Title': paper.get('title', ''),
            'Year': paper.get('year', ''),
            'Journal': paper.get('journal', {}).get('title', ''),
            'DOI': paper.get('doi', ''),
            'First Author': '',
            'First Author Affiliation': '',
            'First Author Country': '',
            'Last Author': '',
            'Last Author Affiliation': '',
            'Last Author Country': '',
            'Funding Country': '',
            'Funder': ''
        }

        # Extract first and last author details
        if paper['authors']:
            first_author = paper['authors'][0]
            last_author = paper['authors'][-1]
            info['First Author'] = first_author.get('first_name', '') + ' ' + first_author.get('last_name', '')
            info['Last Author'] = last_author.get('first_name', '') + ' ' + last_author.get('last_name', '')
            if first_author['affiliations']:
                first_affiliation = first_author['affiliations'][0]
                info['First Author Affiliation'] = first_affiliation.get('name', '')
                info['First Author Country'] = first_affiliation.get('country', '')
            if last_author['affiliations']:
                first_affiliation = first_author['affiliations'][0]
                info['Last Author Affiliation'] = first_affiliation.get('name', '')
                info['Last Author Country'] = first_affiliation.get('country', '')

        try:
            # Extract funding details
            if paper['funders']:
                funder_info = paper['funders'][0]
                info['Funder'] = funder_info.get('name', '')
                if funder_info['country_name']:
                    info['Funding Country'] = funder_info['country_name']
        except:
            info['Founder'] = 'None'
            info['Funding Country'] = 'None'

        return info, paper



# Example PMIDs (replace with your actual PMIDs)
pmids = pd.unique(df['PMID'])

# Fetch details for each PMID
papers_data = []
papers_data_raw = []
no_info = []
i = 0
for pmid in pmids:
    #print(f'Extracting information of PMID: {pmid}')
    try:
      paper_info, info_raw = get_paper_details(pmid)
    except:
      print('No Info...')
      no_info.append(pmid)
      continue

    if paper_info:
        #print(info_raw)
        papers_data.append(paper_info)
        papers_data_raw.append(info_raw)

    if i % 100 == 0:
      print(f'{i} papers extracted')
    i += 1

# Create a DataFrame from the fetched data
papers_df = pd.DataFrame(papers_data_raw)
papers_df_prep = pd.DataFrame(papers_data)

# Display the DataFrame
# papers_df

Returned Publications: 1 (total = 1)
Time: 0.65s
WARNINGS [1]
Field current_organization_id of the authors field is deprecated and will be removed in the next major release.
0 papers extracted
Returned Publications: 1 (total = 1)
Time: 3.72s
WARNINGS [1]
Field current_organization_id of the authors field is deprecated and will be removed in the next major release.
Returned Publications: 1 (total = 1)
Time: 2.21s
WARNINGS [1]
Field current_organization_id of the authors field is deprecated and will be removed in the next major release.
Returned Publications: 1 (total = 1)
Time: 3.82s
WARNINGS [1]
Field current_organization_id of the authors field is deprecated and will be removed in the next major release.
Returned Publications: 1 (total = 1)
Time: 6.04s
WARNINGS [1]
Field current_organization_id of the authors field is deprecated and will be removed in the next major release.
Returned Publications: 1 (total = 1)
Time: 6.03s
WARNINGS [1]
Field current_organization_id of the authors fiel

In [6]:
papers_df.to_csv(f'Global_Health_{str(year)}_raw.csv', index=False)
papers_df_prep.to_csv(f'Global_Health_{str(year)}.csv', index=False)

### Extract the information using batches and then concatenate the files using:

In [6]:
files =[file for file in os.listdir() if 'raw' in file]
files

# Read all csv files in 'files' and concatenate them into one dataframe
data = pd.concat([pd.read_csv(file) for file in files])
# remove duplicates
data = data.drop_duplicates()
data = data.drop_duplicates(subset='id')

data.head()

,authors,date,doi,id,journal,linkout,source_title,title,year,funder_countries,funders,authors_count
0,"[{'affiliations': [{'city': 'Hull', 'city_id':...",2018-12-27,10.1016/j.jpainsymman.2018.12.007,pub.1110927750,"{'id': 'jour.1096258', 'title': 'Journal of Pa...",http://www.jpsmjournal.com/article/S0885392418...,"{'id': 'jour.1096258', 'title': 'Journal of Pa...",A Life or “Good Death” Situation? A Worldwide ...,2018,NaN,NaN,NaN
1,"[{'affiliations': [{'city': 'Baltimore', 'city...",2018-12-29,10.1186/s12909-018-1424-7,pub.1110980386,"{'id': 'jour.1028429', 'title': 'BMC Medical E...",https://bmcmededuc.biomedcentral.com/track/pdf...,"{'id': 'jour.1028429', 'title': 'BMC Medical E...",The ethical experiences of trainees on short-t...,2018,NaN,NaN,NaN
2,"[{'affiliations': [{'city': 'Varese', 'city_id...",2018-12-28,10.1055/s-0038-1676611,pub.1110955745,"{'id': 'jour.1085446', 'title': 'Thrombosis an...",https://www.thieme-connect.com/products/ejourn...,"{'id': 'jour.1085446', 'title': 'Thrombosis an...",Characteristics and Management of Patients wit...,2018,"[{'id': 'DE', 'name': 'Germany'}]","[{'city_name': 'Leverkusen', 'country_code': '...",NaN
3,"[{'affiliations': [{'city': 'Niterói', 'city_i...",2018-12-28,10.1111/scd.12349,pub.1110957443,"{'id': 'jour.1054725', 'title': 'Special Care ...",NaN,"{'id': 'jour.1054725', 'title': 'Special Care ...",Global prevalence of dental caries in athletes...,2018,NaN,NaN,NaN
4,"[{'affiliations': [{'city': 'Hobart', 'city_id...",2018-12-28,10.1161/jaha.118.010235,pub.1110957656,"{'id': 'jour.1046624', 'title': 'Journal of th...",https://doi.org/10.1161/jaha.118.010235,"{'id': 'jour.1046624', 'title': 'Journal of th...",Sex Differences in Severity of Stroke in the I...,2018,"[{'id': 'BR', 'name': 'Brazil'}, {'id': 'AU', ...","[{'acronym': 'CNPq', 'city_name': 'Brasília', ...",NaN


In [9]:
data.to_csv('Global_health_papers_2018_2023.csv', index=False)